In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# plot mean and std data for different machines

# set machine
set_machine = 'OCEAN'  # Options: 'OCEAN', 'DLRA'

# read data
df_mean_std = pd.read_csv(
    f'output/{set_machine}/mean_std_datarecorder_with_all_measures.csv'
)
df_dryness = pd.read_csv(
    f'output/{set_machine}/dryness_data.csv'
)

# axis & hue logic
if set_machine == 'OCEAN':
    x = 't_duration'
    hue = None
elif set_machine == 'DLRA':
    x = 'n_UL'
    hue = 'T_drying'

# create figure
plt.figure(figsize=(10, 6))

# ---- scatter: all raw values (background) ----
sns.scatterplot(
    data=df_dryness,
    x=x,
    y='m_diff',
    hue=hue,
    alpha=0.35,
    legend=True
)

# ---- line: mean ----
sns.lineplot(
    data=df_mean_std,
    x=x,
    y='m_water_mean',
    hue=hue,
    marker='o',
    legend=False  # avoid duplicate legend
)

# ---- shaded std band ----
ax = plt.gca()

if hue:
    hues = df_mean_std[hue].unique()
    colors = {h: ax.lines[i].get_color() for i, h in enumerate(hues)}

    for h in hues:
        grp = df_mean_std[df_mean_std[hue] == h]
        plt.fill_between(
            grp[x],
            grp['m_water_mean'] - grp['m_water_std'],
            grp['m_water_mean'] + grp['m_water_std'],
            alpha=0.25,
            color=colors[h],

        )
else:
    plt.fill_between(
        df_mean_std[x],
        df_mean_std['m_water_mean'] - df_mean_std['m_water_std'],
        df_mean_std['m_water_mean'] + df_mean_std['m_water_std'],
        alpha=0.25
    )

# ---- regression ----
if hue:
    for h, grp in df_dryness.groupby(hue):
        sns.regplot(
            data=grp,
            x=x,
            y='m_diff',
            scatter=False,
            ci=None,
            color=colors[h],
            line_kws={'linestyle': '--', 'linewidth': 2},
            ax=ax,
            order=2
        )
else:
    sns.regplot(
        data=df_dryness,
        x=x,
        y='m_diff',
        scatter=False,
        ci=None,
        line_kws={'linestyle': '--', 'linewidth': 2},
        ax=ax,
        order=2
    )

# ---- labels & styling ----
plt.title(f'Dryness Values with Mean ± Std for {set_machine}')
plt.xlabel('Drying Time (s)' if x == 't_duration' else 'NRotation Speed of UL')
plt.ylabel('Dryness Value')
plt.grid(True)
plt.legend(title='Drying Temperature (T_drying)' if hue else None)

# save & show
plt.savefig(f'output/{set_machine}/mean_std_dryness_plot.png', dpi=300, bbox_inches='tight')
plt.show()


: 